In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = '..'

conn = duckdb.connect(database=':memory:')

MIMIC_ICU_PATH = f"{PROJECT_ROOT}/data/mimic_iv_demo/icu"
MIMIC_HOSP_PATH = f"{PROJECT_ROOT}/data/mimic_iv_demo/hosp"

# Exploration

In [ ]:
icustays_df = pd.read_csv(f'{MIMIC_ICU_PATH}/icustays.csv.gz')
icustays_df.head()

In [ ]:
d_items_df = pd.read_csv(f'{MIMIC_ICU_PATH}/d_items.csv.gz')
item_filter = (d_items_df['label'].str.lower().str.contains("asopressin"))
# item_filter = (d_items_df['itemid'] == 224149)
# item_filter = (d_items_df['linksto'] == "outputevents")
# item_filter = (d_items_df['category'] == "Dialysis")
d_items_df[item_filter]

In [ ]:
labitems_df = pd.read_csv(f'{MIMIC_HOSP_PATH}/d_labitems.csv.gz')
lab_filter = (labitems_df['itemid'] == 50963)
labitems_df[lab_filter]

In [ ]:
chartevents_df = pd.read_csv(f'{MIMIC_ICU_PATH}/chartevents.csv.gz')
chartevents_df[chartevents_df['itemid'] == 220048]["value"].unique()

# Cohort extraction

## Exclusions and target definition

In [ ]:
cohort_query = f"""
WITH first_icu_stays AS (
  SELECT 
    subject_id,
    hadm_id,
    stay_id,
    intime,
    los
  FROM read_csv_auto('{MIMIC_ICU_PATH}/icustays.csv.gz')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY intime ASC) = 1
),

-- BASE FILTER: First ICU stays for patients over 18 that lasted more than 2 days

eligible_adults AS (
  SELECT 
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.intime,
    icu.los,
    adm.race,
    adm.admittime,
    pat.gender,
    (pat.anchor_age + (EXTRACT(YEAR FROM icu.intime) - pat.anchor_year)) AS admission_age
  FROM 
    first_icu_stays icu
  INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/admissions.csv.gz') adm 
    ON icu.hadm_id = adm.hadm_id
  INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/patients.csv.gz') pat 
    ON icu.subject_id = pat.subject_id
  WHERE 
    icu.los > 2.0
    AND (pat.anchor_age + (EXTRACT(YEAR FROM icu.intime) - pat.anchor_year)) >= 18
),

-- EXCLUSION 1: AF billing code in ANY prior admission

historical_af AS (
  SELECT DISTINCT ea.subject_id
  FROM eligible_adults ea
  INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/diagnoses_icd.csv.gz') diag 
    ON ea.subject_id = diag.subject_id
  INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/admissions.csv.gz') adm_past 
    ON diag.hadm_id = adm_past.hadm_id
  WHERE 
    adm_past.admittime < ea.admittime 
    AND (diag.icd_code = '42731' OR diag.icd_code LIKE 'I48%')
),

-- EXCLUSION 2: Early-onset / Pre-existing presentation inside ICU Day 1 window

icu_day1_af AS (
  SELECT DISTINCT ea.stay_id
  FROM eligible_adults ea
  INNER JOIN 
    read_csv_auto('{MIMIC_ICU_PATH}/chartevents.csv.gz') chart 
    ON ea.stay_id = chart.stay_id
  WHERE 
    chart.itemid = 220048
    AND chart.charttime BETWEEN ea.intime AND ea.intime + INTERVAL 1 DAY
    AND (LOWER(chart.value) LIKE '%atrial fibrillation%' OR LOWER(chart.value) LIKE '%a-fib%')
),

-- CASE INCLUSION: New Onset AF after ICU stay Day 1

icu_day2_plus_af AS (
  SELECT DISTINCT ea.stay_id
  FROM eligible_adults ea
  INNER JOIN 
    read_csv_auto('{MIMIC_ICU_PATH}/chartevents.csv.gz') chart 
    ON ea.stay_id = chart.stay_id
  WHERE 
    chart.itemid = 220048
    AND chart.charttime > ea.intime + INTERVAL 1 DAY 
    AND (LOWER(chart.value) LIKE '%atrial fibrillation%' OR LOWER(chart.value) LIKE '%a-fib%')
),

-- Identify any AF billing code assigned to the CURRENT hospital admission
current_adm_af_icd AS (
  SELECT DISTINCT ea.hadm_id
  FROM eligible_adults ea
  INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/diagnoses_icd.csv.gz') diag 
    ON ea.hadm_id = diag.hadm_id
  WHERE 
    (diag.icd_code = '42731' OR diag.icd_code LIKE 'I48%')
),

-- EXCLUSION 3: History of cardiac surgery, including valve surgery and coronary artery bypass grafting

cardiac_surgery AS (
  SELECT DISTINCT ea.subject_id
  FROM eligible_adults ea
  INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/procedures_icd.csv.gz') proc 
    ON ea.subject_id = proc.subject_id
  INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/admissions.csv.gz') adm_proc 
    ON proc.hadm_id = adm_proc.hadm_id
  WHERE 
    adm_proc.admittime <= ea.admittime
    AND (
      -- ICD-9: 361x (CABG), 351x/352x (Valves)
      proc.icd_code LIKE '361%' 
      OR proc.icd_code LIKE '351%' 
      OR proc.icd_code LIKE '352%'
      -- ICD-10-PCS: 021% (Bypass Heart), 02R% (Replace Heart Valve)
      OR proc.icd_code LIKE '021%' 
      OR proc.icd_code LIKE '02R%'
    )
)

-- FINAL SELECTION
SELECT 
  ea.*,
  CASE WHEN t.stay_id IS NOT NULL THEN 1 ELSE 0 END AS target_noaf
FROM eligible_adults ea
LEFT JOIN 
  icu_day2_plus_af t 
  ON ea.stay_id = t.stay_id
WHERE 
  ea.subject_id NOT IN (SELECT subject_id FROM historical_af)
  AND ea.stay_id NOT IN (SELECT stay_id FROM icu_day1_af)
  AND NOT (
    ea.hadm_id IN (SELECT hadm_id FROM current_adm_af_icd) 
    AND t.stay_id IS NULL
  )
  AND ea.subject_id NOT IN (SELECT subject_id FROM cardiac_surgery)
"""

final_cohort = conn.execute(cohort_query).df()

conn.register("cohort", final_cohort)

print(f"Final Cohort: {final_cohort.shape[0]}")
print(f"Ethnic proportions: {final_cohort["race"].value_counts(normalize=True)}")
print(f"Case cohort: {final_cohort['target_noaf'].sum()}")
print(f"Control cohort: {final_cohort.shape[0] - final_cohort['target_noaf'].sum()}")

## Feature extraction

### Vitals

In [ ]:
vitals_query = f"""
SELECT 
    co.stay_id,

    -- Vitals Min/Max

    MIN(CASE WHEN ch.itemid = 220045 THEN ch.valuenum END) AS hr_min,
    MAX(CASE WHEN ch.itemid = 220045 THEN ch.valuenum END) AS hr_max,

    MIN(CASE WHEN ch.itemid = 220210 THEN ch.valuenum END) AS rr_min,
    MAX(CASE WHEN ch.itemid = 220210 THEN ch.valuenum END) AS rr_max,

    MIN(CASE WHEN ch.itemid IN (220179, 220059) THEN ch.valuenum END) AS sbp_min,
    MAX(CASE WHEN ch.itemid IN (220179, 220059) THEN ch.valuenum END) AS sbp_max,

    MIN(CASE WHEN ch.itemid IN (220180, 220060) THEN ch.valuenum END) AS dbp_min,
    MAX(CASE WHEN ch.itemid IN (220180, 220060) THEN ch.valuenum END) AS dbp_max,

    MIN(CASE WHEN ch.itemid IN (223761, 223762) THEN 
                CASE WHEN ch.itemid = 223762 THEN (ch.valuenum - 32) * 5/9 ELSE ch.valuenum END 
        END) AS temp_min,
    MAX(CASE WHEN ch.itemid IN (223761, 223762) THEN 
                CASE WHEN ch.itemid = 223762 THEN (ch.valuenum - 32) * 5/9 ELSE ch.valuenum END 
        END) AS temp_max,

    -- SpO2 (Min Only per Guan et al protocol)
    MIN(CASE WHEN ch.itemid = 220277 THEN ch.valuenum END) AS spo2_min,

    -- Weight Extraction (First valid entry)
    FIRST(CASE WHEN ch.itemid IN (226512, 224639, 226846) THEN ch.valuenum END) AS weight
FROM cohort co
INNER JOIN read_csv_auto('{MIMIC_ICU_PATH}/chartevents.csv.gz') ch ON co.stay_id = ch.stay_id
WHERE ch.charttime BETWEEN co.intime AND co.intime + INTERVAL 1 DAY
GROUP BY co.stay_id;
"""
df_vitals = conn.execute(vitals_query).df()

In [ ]:
df_vitals.head()

### Lab measurements

In [ ]:
labs_query = f"""
SELECT 
    co.stay_id,

    MIN(CASE WHEN le.itemid = 51222 THEN le.valuenum END) AS hemoglobin_min,
    MAX(CASE WHEN le.itemid = 51222 THEN le.valuenum END) AS hemoglobin_max,

    MIN(CASE WHEN le.itemid IN (51300, 51301) THEN le.valuenum END) AS wbc_min,
    MAX(CASE WHEN le.itemid IN (51300, 51301) THEN le.valuenum END) AS wbc_max,
    
    MIN(CASE WHEN le.itemid = 51265 THEN le.valuenum END) AS platelets_min,
    MAX(CASE WHEN le.itemid = 51265 THEN le.valuenum END) AS platelets_max,
    
    MIN(CASE WHEN le.itemid = 51006 THEN le.valuenum END) AS bun_min,
    MAX(CASE WHEN le.itemid = 51006 THEN le.valuenum END) AS bun_max,
    
    MIN(CASE WHEN le.itemid = 50912 THEN le.valuenum END) AS creatinine_min,
    MAX(CASE WHEN le.itemid = 50912 THEN le.valuenum END) AS creatinine_max,
    
    MIN(CASE WHEN le.itemid = 50809 THEN le.valuenum END) AS glucose_min,
    MAX(CASE WHEN le.itemid = 50809 THEN le.valuenum END) AS glucose_max,
    
    MIN(CASE WHEN le.itemid = 50868 THEN le.valuenum END) AS aniongap_min,
    MAX(CASE WHEN le.itemid = 50868 THEN le.valuenum END) AS aniongap_max,
    
    MIN(CASE WHEN le.itemid IN (50971, 50822) THEN le.valuenum END) AS potassium_min,
    MAX(CASE WHEN le.itemid IN (50971, 50822) THEN le.valuenum END) AS potassium_max,
    
    MIN(CASE WHEN le.itemid IN (50983, 50824) THEN le.valuenum END) AS sodium_min,
    MAX(CASE WHEN le.itemid IN (50983, 50824) THEN le.valuenum END) AS sodium_max,
    
    MIN(CASE WHEN le.itemid = 50893 THEN le.valuenum END) AS calcium_min,
    MAX(CASE WHEN le.itemid = 50893 THEN le.valuenum END) AS calcium_max,
    
    MIN(CASE WHEN le.itemid = 50910 THEN le.valuenum END) AS ck_cpk_min,
    MAX(CASE WHEN le.itemid = 50910 THEN le.valuenum END) AS ck_cpk_max,
    
    MIN(CASE WHEN le.itemid = 50911 THEN le.valuenum END) AS ck_mb_min,
    MAX(CASE WHEN le.itemid = 50911 THEN le.valuenum END) AS ck_mb_max,
    
    MIN(CASE WHEN le.itemid = 50963 THEN le.valuenum END) AS ntbnp_min,
    MAX(CASE WHEN le.itemid = 50963 THEN le.valuenum END) AS ntbnp_max
FROM cohort co
INNER JOIN 
    read_csv_auto('{MIMIC_HOSP_PATH}/labevents.csv.gz') le 
    ON co.hadm_id = le.hadm_id
WHERE le.charttime BETWEEN co.intime AND co.intime + INTERVAL 1 DAY
GROUP BY co.stay_id;
"""
df_labs = conn.execute(labs_query).df()

### Urine output

In [ ]:
urine_query = f"""
WITH raw_urine_flows AS (
  SELECT 
      co.stay_id,

      SUM(CASE WHEN oe.itemid IN (226559, 226560, 226561, 226563, 226564, 226565, 226567, 226584) THEN oe.value ELSE 0 END) AS standard_urine,
      
      -- Gross Irrigant/Urine combination out
      SUM(CASE WHEN oe.itemid = 227489 THEN oe.value ELSE 0 END) AS gross_irrigant_out,
      
      -- Active Irrigant volume fluid in (to be subtracted)
      SUM(CASE WHEN oe.itemid = 227488 THEN oe.value ELSE 0 END) AS irrigant_in
  FROM cohort co
  INNER JOIN 
    read_csv_auto('{MIMIC_ICU_PATH}/outputevents.csv.gz') oe 
    ON co.stay_id = oe.stay_id
  WHERE oe.charttime BETWEEN co.intime AND co.intime + INTERVAL 1 DAY
  GROUP BY co.stay_id
)
SELECT 
  stay_id,
  -- Net urine calculation: Standard output + (Gross Drain Out - Infused Saline In)
  GREATEST(0, standard_urine + (gross_irrigant_out - irrigant_in)) AS urine_output_total
FROM 
  raw_urine_flows;
"""

df_urine = conn.execute(urine_query).df()

### Interventions

In [ ]:
interventions_query = f"""
SELECT 
    co.stay_id,

    -- 1. Mechanical Ventilation (Invasive or Non-Invasive Status)
    CASE WHEN EXISTS (
        SELECT 1 FROM read_csv_auto('{MIMIC_ICU_PATH}/procedureevents.csv.gz') pe 
        WHERE pe.stay_id = co.stay_id 
          AND pe.itemid IN (225792, 225794, 224385)
          AND pe.starttime <= co.intime + INTERVAL 1 DAY
    ) THEN 1 ELSE 0 END AS vent_day1,
    
    -- 2. Continuous Renal Replacement Therapy (CRRT)
    CASE WHEN EXISTS (
        SELECT 1 FROM read_csv_auto('{MIMIC_ICU_PATH}/procedureevents.csv.gz') pe 
        WHERE pe.stay_id = co.stay_id 
          AND pe.itemid IN (225441, 225802, 225803, 225805, 224149, 225809)
          AND pe.starttime <= co.intime + INTERVAL 1 DAY
    ) THEN 1 ELSE 0 END AS crrt_day1,
    
    -- 3. Vasopressors (Continuous Infusions: Norepi, Epi, Phenylepi, Vasopressin, Dopamine)
    CASE WHEN EXISTS (
        SELECT 1 FROM read_csv_auto('{MIMIC_ICU_PATH}/inputevents.csv.gz') ie 
        WHERE ie.stay_id = co.stay_id 
          AND ie.itemid IN (221906, 229617, 221289, 221749, 222315, 221662)
          AND ie.starttime <= co.intime + INTERVAL 1 DAY
    ) THEN 1 ELSE 0 END AS vasopressors_day1,
    
    -- 4. Antibiotics 
    -- TO BE DONE ON THE FULL DB WITH MIMIC CODES

FROM cohort co;
"""
df_interventions = conn.execute(interventions_query).df()

### Comorbidities

In [ ]:
comorbidities_query = f"""
WITH cohort_diagnoses AS (
  SELECT 
    co.hadm_id,
    REPLACE(diag.icd_code, '.', '') AS clean_code,
    diag.icd_version
  FROM cohort co
  INNER JOIN read_csv_auto('{MIMIC_HOSP_PATH}/diagnoses_icd.csv.gz') diag 
    ON co.hadm_id = diag.hadm_id
)
SELECT 
  hadm_id,

  -- 1. Hypertension
  MAX(CASE WHEN (icd_version = 10 
          AND clean_code IN ('I10', 'I11', 'I12', 'I13', 'I15', 'N262'))
      OR (icd_version = 9 
          AND clean_code IN ('4011', '4019', '40210', '40290', '40410', '40490', '40511', '40519', '40591', '40599')) 
      THEN 1 ELSE 0 END) AS comorb_hypertension,

  -- 2. Diabetes Mellitus
  MAX(CASE WHEN (icd_version = 10 
          AND LEFT(clean_code, 3) IN ('E10', 'E11', 'E12', 'E13', 'E14')
      OR (icd_version = 9 
          AND clean_code LIKE '250%'))
      THEN 1 ELSE 0 END) AS comorb_diabetes,
  
  -- 3. Chronic Kidney Disease (CKD): Anchored to N18 and 585
  MAX(CASE WHEN (icd_version = 10 
          AND clean_code LIKE 'N18%') 
      OR (icd_version = 9 
          AND clean_code LIKE '585%') 
      THEN 1 ELSE 0 END) AS comorb_ckd,

  -- 4. Old Myocardial Infarction (MI)
  MAX(CASE WHEN (icd_version = 10 
          AND clean_code = 'I252') 
      OR (icd_version = 9 
          AND clean_code = '412')
      THEN 1 ELSE 0 END) AS comorb_mi,
              
  -- 5. Chronic Lung Disease
  MAX(CASE WHEN (icd_version = 10 
          AND LEFT(clean_code, 3) IN ('J41', 'J42', 'J43', 'J44')) 
      OR (icd_version = 9 
          AND LEFT(clean_code, 3) IN ('491', '492', '496'))
      THEN 1 ELSE 0 END) AS comorb_lung,
              
  -- 6. Chronic Liver Disease
  MAX(CASE WHEN (icd_version = 10 
          AND (clean_code LIKE 'K70%' OR clean_code LIKE 'K74%')) 
      OR (icd_version = 9 
          AND (clean_code LIKE '571%' OR clean_code LIKE '572%')) 
      THEN 1 ELSE 0 END) AS comorb_liver,

  -- 7. Alcohol use
  MAX(CASE WHEN (icd_version = 10 
          AND (clean_code IN ('V113', 'E52', 'G621', 'I426', 'K292', 'T51', 'Z714', 'Z658') 
              OR LEFT(clean_code, 3) IN ('F10', 'K70'))) 
      OR (icd_version = 9 
          AND (clean_code IN ('30393', '30503') OR LEFT(clean_code, 3) = '291')) 
      THEN 1 ELSE 0 END) AS comorb_alcohol

FROM cohort_diagnoses
GROUP BY hadm_id;
"""

df_comorbidities = conn.execute(comorbidities_query).df()

# Full cohort and features

In [ ]:
dataset = final_cohort.merge(df_vitals, on="stay_id", how="left")
dataset = dataset.merge(df_labs, on="stay_id", how="left")
dataset = dataset.merge(df_urine, on="stay_id", how="left")
dataset = dataset.merge(df_interventions, on="stay_id", how="left")
dataset = dataset.merge(df_comorbidities, on="hadm_id", how="left")

dataset.describe().T